# BiLSTM peptide classifier

Standalone RNN workflow. It imports only modules in `RNN`; the input CSV is read from the Kaggle dataset path. The model uses peptide sequence plus protein-derived flanks, start/end coordinates, and eight biological descriptors. B-cell uses stratified 5-fold protein-level cross-validation; SARS uses stratified peptide-level 5-fold cross-validation because the current SARS dataset has one protein. SARS folds share a source protein and do not measure performance on unseen proteins. Tabular imputation/scaling is fit on each fold's training rows only.

In [ ]:
from pathlib import Path
import torch
from RNN.CFG import CFG
from RNN.train import run_training

cfg = CFG()
cfg.DATASET_NAME = "bcell"
cfg.DATA_DIR = Path("/kaggle/input/datasets/daffamazhar/b-cell-epitope")  # adjust to your Kaggle input path
cfg.OUTPUT_DIR = Path("/kaggle/working/RNN-artifacts")
cfg.MAX_PEPTIDE_LENGTH = 20
cfg.FLANKING_SIZE = 5
cfg.VALIDATION_SIZE = 0.2
cfg.RANDOM_STATE = 42
cfg.BATCH_SIZE = 64
cfg.EPOCHS = 100
cfg.DEVICE = "auto"

results = run_training(cfg)
from IPython.display import display
import pandas as pd

print("Split strategy:", results["split_strategy"])
print("Validation interpretation:", results["validation_metrics_interpretation"])
print("Pooled out-of-fold metrics:")
display(pd.Series(results["validation_metrics"], name="value").to_frame())
print("Mean/std across folds:")
display(pd.DataFrame({"mean": results["mean_validation_metrics"], "std": results["std_validation_metrics"]}))
print("Per-fold training and evaluation metrics:")
display(pd.DataFrame([
    {
        "fold": item["fold"],
        **{f"training_{name}": value for name, value in item["training_metrics"].items()},
        **{f"evaluation_{name}": value for name, value in item["validation_metrics"].items()},
    }
    for item in results["fold_metrics"]
]))